# BM25 Lexical Search

Revisiting the Previous lesson

In [1]:
from utils.util_funcs import *
from utils.vector_index import VectorIndex
from utils.bm25 import BM25Index
from dotenv import load_dotenv

import voyageai

load_dotenv()

voyage_client = voyageai.Client()

Initialised <anthropic.Anthropic object at 0x112a04050> with the model: claude-haiku-4-5


/Users/aidanlockwood/Library/CloudStorage/OneDrive-BluPanteraTechPtyLtd/GitHub/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
with open("./supporting_materials/report.md", "r") as f:
    text = f.read()

chunks = chunk_by_section(text)

chunks[3]

'Methodology\n\nThe insights compiled within this Annual Interdisciplinary Research Review represent a synthesis of findings drawn from standard departmental reporting cycles, specialized project updates, and cross-functional review meetings conducted throughout the year. Data sources included internal project databases, laboratory notebooks, financial reporting systems, legal case summaries, security incident logs, and minutes from dedicated working groups. A central review committee, comprising representatives nominated by each division head, was tasked with identifying key developments and potential cross-domain implications. This committee utilized a standardized reporting template to capture essential details, including unique identifiers (project codes, error numbers, case references, etc.) and progress metrics. Subsequent analysis focused on identifying thematic overlaps, shared challenges, and opportunities for synergistic development, forming the basis of this consolidated rep

## Step 2: Generate Embeddings

In [3]:
embeddings = generate_embedding(chunks)

## Step 3: Storing Embeddings into Vector Database

In [4]:
store = VectorIndex()

for embedding, chunk in zip(embeddings, chunks):
    store.add_vector(embedding, { 
        'content': chunk
    })

## Step 4: Embedding User Query
User will ask a question. Generate an embedding for it

In [5]:
user_embedding = generate_embedding("What happened with INC-2023-Q4-011?")

In [6]:
results = store.search(user_embedding, 2)

for doc, distance in results: 
    print(distance, '\n', doc['content'][0:200], '\n')

0.48256669589940315 
 Section 10: Cybersecurity Analysis - Incident Response Report: INC-2023-Q4-011

The Cybersecurity Operations Center successfully contained and remediated a targeted intrusion attempt tracked as `INC-2 

0.5099757171382763 
 Section 2: Software Engineering - Project Phoenix Stability Enhancements

The Software Engineering division dedicated considerable effort to improving the stability and performance of the core systems 



Noting that the relevance score is higher on Section 2 than Section 10, despite the fact that Section 10 is directly referring to the incident

To help solve this problem, the user's query will use both Semantic Search and Lexical Search (classic text search) to better respond to the user's query.

**This is achieved with the BM25 algorithm**

## Step 1: Tokenise the User Query

## Step 2: Find the Frequency of Each Word in All Documents

In [7]:
store = BM25Index()

for chunk in chunks: 
    store.add_document({
        'content': chunk
    })

## Step 3: Terms used Less Frequently given Higher Weighting in Search

In [ ]:
results = store.search('What happened with INC-2023-Q4-011?', 3)

for doc, distance in results:
    print(distance, '\n', doc['content'][0:200], '\n---\n')

## Step 4: Find the Text Chunk with the Higher Weighted Chunk